# 02 · Chấm điểm + Leaderboard + Biểu đồ đánh đổi

Chạy sau khi đã có hypotheses của các variant. Không cần GPU. Chấm mọi cặp (variant, dataset).


In [ ]:
%cd /content/benchmark-for-asr-model
!git pull -q
!pip install -q -r requirements/base.txt

In [ ]:
import glob, os
hyps = [p for p in glob.glob('results/hypotheses/*/*.jsonl') if not p.endswith('.meta.json')]
print(len(hyps), 'hypotheses files')
for h in hyps:
    model = os.path.basename(os.path.dirname(h)); ds = os.path.basename(h)[:-6]
    !PYTHONPATH=src python -m benchmark.runners.run_scoring \
        --manifest data/manifests/{ds}.jsonl --hyp {h} \
        --norm-config configs/normalization/vi.yaml --model {model} \
        --out results/metrics/{model}/{ds}.json
    !PYTHONPATH=src python -m benchmark.report.error_analysis \
        --manifest data/manifests/{ds}.jsonl --hyp {h} \
        --norm-config configs/normalization/vi.yaml --model {model} --top 25 \
        --out-md results/reports/errors_{model}_{ds}.md

### Leaderboard


In [ ]:
!PYTHONPATH=src python -m benchmark.report.build_report \
  --metrics-dir results/metrics \
  --out-md results/reports/leaderboard.md --out-csv results/reports/leaderboard.csv
from IPython.display import Markdown, display
display(Markdown(open('results/reports/leaderboard.md').read()))

### Biểu đồ đánh đổi độ chính xác ↔ tốc độ


In [ ]:
!PYTHONPATH=src python -m benchmark.report.plot_tradeoff \
  --csv results/reports/leaderboard.csv --out results/reports/tradeoff.png
from IPython.display import Image
Image('results/reports/tradeoff.png')